# 03 — RAG: buscar antes de responder

No notebook 01 vimos que não dá para colar as ~1.050 transações no prompt.
O **RAG** (*Retrieval-Augmented Generation*) resolve isso em 3 passos:

1. **Indexar** (uma vez): cada transação vira texto, e cada texto vira um **vetor** (embedding).
2. **Buscar** (a cada pergunta): a pergunta também vira vetor; pegamos as transações com vetores mais parecidos.
3. **Gerar**: colamos só essas transações no prompt e o LLM responde.

Tudo isso está em `src/faturas/rag.py`, em poucas linhas de numpy — sem framework.

> Pré-requisito: `ollama pull nomic-embed-text`

In [ ]:
import time
from pathlib import Path

import numpy as np

from faturas.dados import carregar_faturas
from faturas.rag import (
    PREFIXO_DOCUMENTO,
    buscar,
    carregar_ou_criar_indice,
    embedar,
    responder,
    transacao_para_texto,
)

df = carregar_faturas("../data")
exemplo = df[df["tipo"] == "gasto"].iloc[-1]["title"]  # sua compra mais recente

## O que é um embedding?

Um **embedding** é uma lista de números (aqui, 768) que representa o *significado* de um texto.
Textos com significado parecido ficam com vetores próximos.

Como `embedar` devolve vetores de tamanho 1, o produto escalar entre dois deles é a
**similaridade de cosseno**: 1 = mesma direção, 0 = nada a ver.

In [ ]:
def matriz_similaridade(palavras):
    v = embedar(palavras, PREFIXO_DOCUMENTO)
    sim = v @ v.T
    for i, p in enumerate(palavras):
        print(f"{p:>12}: " + "  ".join(f"{sim[i, j]:.2f}" for j in range(len(palavras))))
    return v


v = matriz_similaridade(["padaria", "pão francês", "gasolina"])
print()
print("formato de cada embedding:", v.shape)

Esperaríamos "padaria" bem mais perto de "pão francês" do que de "gasolina"...
mas os números ficam quase iguais. Compare com as mesmas palavras em inglês:

In [ ]:
matriz_similaridade(["bakery", "bread", "gasoline"]);

O `nomic-embed-text` foi treinado principalmente em **inglês** — em português ele
distingue pouco os significados. Guarde isso: vai aparecer na busca logo abaixo.

## Passo 1 — Indexar

Cada transação vira um "documento" de uma linha:

In [ ]:
for _, linha in df.tail(5).iterrows():
    print(transacao_para_texto(linha))

Agora geramos o embedding de todas. Sem cache, isso leva cerca de 1 minuto, então
o resultado fica salvo em `data/embeddings.npz` (fora do git). Na segunda vez é instantâneo —
e se você adicionar uma fatura nova, o cache percebe e recalcula.

In [ ]:
cache = Path("../data/embeddings.npz")
t = time.time()
indice = carregar_ou_criar_indice(df, cache)
print(f"1ª chamada: {time.time() - t:.1f}s")
t = time.time()
indice = carregar_ou_criar_indice(df, cache)
print(f"2ª chamada: {time.time() - t:.2f}s  ->  {indice.vetores.shape[0]} vetores de {indice.vetores.shape[1]} números")

## Passo 2 — Buscar

A busca é só: `similaridades = vetores @ vetor_da_pergunta`, e pegar os maiores.

In [ ]:
for pergunta in ["restaurante", exemplo, "assinatura mensal"]:
    print(f"\n🔎 {pergunta}")
    for texto in buscar(pergunta, indice, k=5):
        print("   ", texto)

Repare nos limites da busca:

- **Nomes exatos** (o estabelecimento de exemplo) funcionam bem.
- **Conceitos** ("assinatura mensal") funcionam mal: o título de uma assinatura (algo como `Streaming*Plano`) não diz
  "assinatura" em lugar nenhum — e, como vimos, o modelo é fraco em português.

Melhorias possíveis (bons exercícios): um modelo multilíngue como `bge-m3`, ou enriquecer
o texto de cada transação com uma categoria antes de embedar.

## Passo 3 — Gerar a resposta

`responder` cola as 10 transações mais parecidas no prompt, com a instrução de usar **só** elas.

In [ ]:
for pergunta in [
    f"Quando comprei no {exemplo} e quanto paguei?",
    "Quais gastos eu tive com pet?",
    "Tenho gastos em farmácia parcelados?",
]:
    print(f"❓ {pergunta}\n{responder(pergunta, indice)}\n")

## Onde o RAG quebra: perguntas de agregação

Agora uma pergunta que precisa de **todas** as transações de um ano:

In [ ]:
pergunta = "Quanto gastei no total em 2025?"
print(responder(pergunta, indice))

In [ ]:
real = df[(df["tipo"] == "gasto") & (df["date"].dt.year == 2025)]["amount"].sum()
print(f"Valor real (pandas): R$ {real:,.2f}".replace(",", "_").replace(".", ",").replace("_", "."))

A resposta do RAG está **errada** — e não por culpa do LLM:

1. a busca só entregou **10** das centenas de transações de 2025;
2. mesmo que entregasse todas, LLMs pequenos não são calculadoras confiáveis.

**Lição:** RAG é ótimo para *encontrar* informação ("quando", "onde", "qual"),
mas ruim para *calcular* sobre muitos registros ("quanto no total", "qual mês mais").

Para isso, o LLM precisa **chamar código** que faz a conta de verdade — é o *tool calling*,
assunto do notebook 04.